# 13: Final validation ranking

Rol: ranking final usando solo validacion artificial.

Carga todos los resultados disponibles y arma una tabla comparable. No usa test oficial. El criterio final es: menor C-MAPSS score, luego menor RMSE, luego menor dangerous_error_pct y MAE como metrica secundaria.


In [1]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "FD001"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
VALIDATION_RANDOM_STATE = 42
CUT_RULS = (20, 50, 80, 110, 140)
BASE_WINDOW_SIZE = 30
BASE_RUL_CAP = 125

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [2]:
from src.fd001_experiment_utils import (
    infer_window_size,
    load_if_exists,
    mlp_seed_summary_as_row,
    normalize_rows,
)

METRIC_COLUMNS = ["mae", "rmse", "r2", "cmapss_score", "cmapss_score_mean", "dangerous_error_pct"]
OUTPUT_PATH = RESULTS_DIR / "fd001_final_validation_ranking.csv"


In [3]:
rows = []

current = load_if_exists(RESULTS_DIR, "fd001_current_cycle_metrics.csv")
if current is not None:
    subset = current.loc[current["model_name"].isin(["Ridge", "RandomForestRegressor"])].copy()
    rows.extend(normalize_rows(subset, "current_cycle_baseline", "Ridge/RF baseline current-cycle"))

temporal = load_if_exists(RESULTS_DIR, "fd001_temporal_metrics.csv")
if temporal is not None:
    subset = temporal.loc[temporal["model_name"].isin(["Ridge", "RandomForestRegressor"])].copy()
    rows.extend(normalize_rows(subset, "temporal_baseline", "Ridge/RF baseline temporal_w30"))

boosting = load_if_exists(RESULTS_DIR, "fd001_boosting_metrics.csv")
if boosting is not None:
    subset = boosting.loc[boosting["model_name"].isin(["LGBMRegressor", "XGBRegressor", "RandomForestRegressor"])].copy()
    rows.extend(normalize_rows(subset, "boosting_discovery", "Boosting discovery at temporal_w30/rul_cap125"))

mlp = load_if_exists(RESULTS_DIR, "fd001_mlp_metrics.csv")
if mlp is not None:
    subset = mlp.loc[mlp["model_name"].isin(["MLP_tabular_cfg_03"])].copy()
    rows.extend(normalize_rows(subset, "mlp_initial_search", "Historical best MLP cfg03 from initial search"))

window_cap = load_if_exists(RESULTS_DIR, "fd001_lgbm_window_cap_search.csv")
if window_cap is not None:
    best_lgbm = window_cap.sort_values(["cmapss_score", "rmse", "dangerous_error_pct", "mae"]).head(1).copy()
    rows.extend(normalize_rows(best_lgbm, "lgbm_window_cap_search", "Best LGBM window/cap by validation criterion"))

mlp_summary = load_if_exists(RESULTS_DIR, "fd001_mlp_cfg03_seed_summary.csv")
if mlp_summary is not None:
    rows.append(mlp_seed_summary_as_row(mlp_summary))

sample_weight = load_if_exists(RESULTS_DIR, "fd001_lgbm_sample_weight_search.csv")
if sample_weight is not None:
    rows.extend(normalize_rows(sample_weight, "lgbm_sample_weight_search", "LGBM sample-weight variants"))

ranking = pd.DataFrame(rows)
for col in METRIC_COLUMNS:
    ranking[col] = pd.to_numeric(ranking[col], errors="coerce")
ranking = ranking.dropna(subset=["cmapss_score", "rmse", "dangerous_error_pct", "mae"]).copy()
ranking = ranking.sort_values(["cmapss_score", "rmse", "dangerous_error_pct", "mae"]).reset_index(drop=True)
ranking.to_csv(OUTPUT_PATH, index=False)

display(ranking)
print("Candidato recomendado por validacion artificial:")
display(ranking.iloc[0][["model_name", "representation", "window_size", "rul_cap", "sample_weight_scheme", "mae", "rmse", "cmapss_score", "dangerous_error_pct", "selection_notes"]])
print(OUTPUT_PATH)


,source,model_name,representation,window_size,rul_cap,sample_weight_scheme,selection_notes,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct
0,lgbm_sample_weight_search,LGBMRegressor,temporal_w50,50.0,125,bin_weights,LGBM sample-weight variants,11.194267,14.037945,0.889380,251.554310,2.540953,7.070707
1,lgbm_sample_weight_search,LGBMRegressor,temporal_w50,50.0,125,aggressive,LGBM sample-weight variants,10.888389,14.046041,0.889252,253.632055,2.561940,7.070707
2,lgbm_sample_weight_search,LGBMRegressor,temporal_w50,50.0,125,soft,LGBM sample-weight variants,11.061073,14.253374,0.885959,260.274518,2.629036,7.070707
3,lgbm_window_cap_search,LGBMRegressor,temporal_w50,50.0,125,none,Best LGBM window/cap by validation criterion,11.297658,14.428233,0.883144,261.005416,2.636418,6.060606
4,lgbm_sample_weight_search,LGBMRegressor,temporal_w50,50.0,125,none,LGBM sample-weight variants,11.297658,14.428233,0.883144,261.005416,2.636418,6.060606
5,boosting_discovery,LGBMRegressor,temporal_w30,30.0,125,none,Boosting discovery at temporal_w30/rul_cap125,12.809901,16.517134,0.846858,397.315013,4.013283,11.111111
6,boosting_discovery,XGBRegressor,temporal_w30,30.0,125,none,Boosting discovery at temporal_w30/rul_cap125,13.931649,17.238230,0.833194,418.575570,4.228036,9.090909
7,mlp_initial_search,MLP_tabular_cfg_03,temporal_w30,30.0,125,none,Historical best MLP cfg03 from initial search,12.626827,17.051460,0.836789,449.529187,4.540699,10.101010
8,boosting_discovery,RandomForestRegressor,temporal_w30,30.0,125,none,Boosting discovery at temporal_w30/rul_cap125,13.817341,17.912073,0.819898,580.933845,5.868019,14.141414
9,temporal_baseline,RandomForestRegressor,temporal_w30,30.0,125,none,Ridge/RF baseline temporal_w30,13.817341,17.912073,0.819898,580.933845,5.868019,14.141414


Candidato recomendado por validacion artificial:


model_name                            LGBMRegressor
representation                         temporal_w50
window_size                                    50.0
rul_cap                                         125
sample_weight_scheme                    bin_weights
mae                                       11.194267
rmse                                      14.037945
cmapss_score                              251.55431
dangerous_error_pct                        7.070707
selection_notes         LGBM sample-weight variants
Name: 0, dtype: object

C:\Users\lauta\OneDrive\Escritorio\ML\TPF-ML\results\FD001\fd001_final_validation_ranking.csv


## Interpretacion

El primer puesto queda elegido por validacion artificial, priorizando el score asimetrico C-MAPSS. Si un modelo mejora RMSE pero empeora mucho errores peligrosos, la decision debe discutirse con la tabla completa, no solo con RMSE.
